In [1]:
import torch
import torch.nn as nn
import numpy as np
import os
import json
import pickle
import pandas as pd
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Cell 2: Dataset, collate, dataloader
class SyntheaLOSDataset(Dataset):
    def __init__(self, sequences_path, encounter_ids=None):
        with open(sequences_path, 'rb') as f:
            self.sequences = pickle.load(f)
        if encounter_ids is not None:
            self.ids = [eid for eid in encounter_ids if eid in self.sequences]
        else:
            self.ids = list(self.sequences.keys())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        eid = self.ids[idx]
        seq = self.sequences[eid]
        X        = torch.tensor(seq['X'], dtype=torch.float32)
        Y        = torch.tensor(seq['Y'], dtype=torch.float32)
        guidance = X[:, -2:].clone()
        length   = torch.tensor(X.shape[0], dtype=torch.long)
        return X, guidance, Y, length, eid


def collate_fn(batch):
    X_list, g_list, Y_list, lengths, eids = zip(*batch)
    X_pad   = pad_sequence(X_list, batch_first=True, padding_value=0.0)
    g_pad   = pad_sequence(g_list, batch_first=True, padding_value=0.0)
    Y_pad   = pad_sequence(Y_list, batch_first=True, padding_value=0.0)
    lengths = torch.stack(lengths)
    return X_pad, g_pad, Y_pad, lengths, list(eids)


def get_dataloaders(sequences_path, val_frac=0.10, test_frac=0.15,
                    batch_size=32, num_workers=2, seed=42):
    rng = np.random.default_rng(seed)
    with open(sequences_path, 'rb') as f:
        all_ids = list(pickle.load(f).keys())
    all_ids = np.array(all_ids)
    rng.shuffle(all_ids)
    n        = len(all_ids)
    n_test   = int(n * test_frac)
    n_val    = int(n * val_frac)
    test_ids  = all_ids[:n_test].tolist()
    val_ids   = all_ids[n_test:n_test + n_val].tolist()
    train_ids = all_ids[n_test + n_val:].tolist()
    print(f"Split — train: {len(train_ids)}, val: {len(val_ids)}, test: {len(test_ids)}")

    def make_loader(ids, shuffle):
        ds = SyntheaLOSDataset(sequences_path, encounter_ids=ids)
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          num_workers=num_workers, collate_fn=collate_fn)

    return (make_loader(train_ids, shuffle=True),
            make_loader(val_ids,   shuffle=False),
            make_loader(test_ids,  shuffle=False),
            train_ids, val_ids, test_ids)

In [3]:
#Cell 3: Model
class MLPEncoder(nn.Module):
    def __init__(self, input_dim, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class RAIM(nn.Module):
    def __init__(self, input_dim, hidden_size=128, guidance_dim=2):
        super().__init__()
        self.hidden_size  = hidden_size
        self.guidance_dim = guidance_dim
        self.encoder      = MLPEncoder(input_dim, hidden_size)
        self.attn_mlp     = nn.Sequential(
            nn.Linear(hidden_size + guidance_dim, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1),
        )
        self.gru_cell         = nn.GRUCell(hidden_size, hidden_size)
        self.regression_head  = nn.Linear(hidden_size, 1)
        self.softmax          = nn.Softmax(dim=1)

    def forward(self, x, guidance, lengths):
        batch_size, T_max, _ = x.shape
        device  = x.device
        encoded = self.encoder(x)
        h       = torch.zeros(batch_size, self.hidden_size, device=device)
        preds   = []

        for t in range(T_max):
            enc_so_far = encoded[:, :t+1, :]
            g_so_far   = guidance[:, :t+1, :]
            g_masked   = g_so_far.clone()
            if t > 0:
                prior_sum = guidance[:, :t, :].sum(dim=1)
                zero_mask = (prior_sum.sum(dim=1) == 0)
                if zero_mask.any():
                    g_masked[zero_mask, :t, :] = 1.0
            g_masked[:, t, :] = 1.0
            attn_input = torch.cat([enc_so_far, g_masked], dim=2)
            scores     = self.attn_mlp(attn_input).squeeze(-1)
            weights    = self.softmax(scores)
            context    = (enc_so_far * weights.unsqueeze(-1)).sum(dim=1)
            h          = self.gru_cell(context, h)
            pred_t     = self.regression_head(h).squeeze(-1)
            preds.append(pred_t)

        return torch.stack(preds, dim=1)

In [4]:
#Cell 4: Loss and epoch
def masked_mse(preds, targets, lengths):
    T_max = preds.shape[1]
    mask  = torch.arange(T_max, device=lengths.device).unsqueeze(0) < lengths.unsqueeze(1)
    return (((preds - targets) ** 2) * mask).sum() / mask.sum()

def masked_mae(preds, targets, lengths):
    T_max = preds.shape[1]
    mask  = torch.arange(T_max, device=lengths.device).unsqueeze(0) < lengths.unsqueeze(1)
    return (torch.abs(preds - targets) * mask).sum() / mask.sum()


def run_epoch(model, loader, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_mse, total_mae, total_windows = 0.0, 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X, guidance, Y, lengths, _ in loader:
            X, guidance, Y, lengths = X.to(device), guidance.to(device), Y.to(device), lengths.to(device)
            preds = model(X, guidance, lengths)
            loss  = masked_mse(preds, Y, lengths)
            mae   = masked_mae(preds, Y, lengths)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
            n_windows      = lengths.sum().item()
            total_mse     += loss.item() * n_windows
            total_mae     += mae.item()  * n_windows
            total_windows += n_windows
    return total_mse / total_windows, total_mae / total_windows

In [5]:
#Cell 5: Train function
def train_run(sequences_path, input_dim, save_dir, label,
              hidden_size=128, guidance_dim=2,
              batch_size=32, epochs=50, lr=1e-3, patience=8,
              val_frac=0.10, test_frac=0.15, seed=42):

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs(save_dir, exist_ok=True)

    train_loader, val_loader, test_loader, _, _, _ = get_dataloaders(
        sequences_path, val_frac=val_frac, test_frac=test_frac,
        batch_size=batch_size, seed=seed
    )

    model     = RAIM(input_dim=input_dim, hidden_size=hidden_size, guidance_dim=guidance_dim).to(device)
    optimizer = Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    best_val_mse, patience_counter, history = float('inf'), 0, []

    print(f"\n{'='*50}\nRunning: {label}  |  input_dim={input_dim}\n{'='*50}")

    for epoch in range(1, epochs + 1):
        train_mse, train_mae = run_epoch(model, train_loader, optimizer, device, train=True)
        val_mse,   val_mae   = run_epoch(model, val_loader,   optimizer, device, train=False)
        scheduler.step(val_mse)
        train_rmse, val_rmse = np.sqrt(train_mse), np.sqrt(val_mse)
        print(f"Epoch {epoch:03d} | Train RMSE: {train_rmse:.4f}  MAE: {train_mae:.4f} | Val RMSE: {val_rmse:.4f}  MAE: {val_mae:.4f}")
        history.append({'epoch': epoch, 'train_rmse': train_rmse, 'train_mae': train_mae,
                        'val_rmse': val_rmse, 'val_mae': val_mae})
        if val_mse < best_val_mse:
            best_val_mse     = val_mse
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(save_dir, 'best_raim.pt'))
            print(f"  -> Saved best (val RMSE: {val_rmse:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    with open(os.path.join(save_dir, 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    # Test eval
    model.load_state_dict(torch.load(os.path.join(save_dir, 'best_raim.pt')))
    test_mse, test_mae = run_epoch(model, test_loader, None, device, train=False)
    test_rmse = np.sqrt(test_mse)
    print(f"\n{label} — Test RMSE: {test_rmse:.4f} days  |  MAE: {test_mae:.4f} days")
    return {'label': label, 'test_rmse': test_rmse, 'test_mae': test_mae}

In [6]:
import os
import pickle

# Find your pkl files
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f.endswith('.pkl'):
            print(os.path.join(root, f))

/content/drive/MyDrive/RAIM_Project_v2/run_05_age_med_years_med/sequences/patient_sequences.pkl
/content/drive/MyDrive/RAIM_Project_v2/run_05_age_med_years_med/matrices/vitals_matrices.pkl
/content/drive/MyDrive/RAIM_Project_v2/run_05_age_med_years_med/matrices/guidance_matrices.pkl
/content/drive/MyDrive/RAIM_Project_v2/run_05_age_med_years_med/matrices/llm_icd_codes.pkl
/content/drive/MyDrive/RAIM_Project_v2/run_05_age_med_years_med/matrices/encounter_notes.pkl
/content/drive/MyDrive/RAIM_Project_v2/run_05_age_med_years_med/matrices/sequences_enriched.pkl
/content/drive/MyDrive/RAIM_Project_v2/run_05_age_med_years_med/matrices/sequences.pkl
/content/drive/MyDrive/RAIM_Project_v2/run_05_age_med_years_med/matrices/sequences_baseline.pkl


In [7]:
#Cell 6: Config — update your paths here
BASE_DIR = '/content/drive/MyDrive/RAIM_Project_v2/run_05_age_med_years_med/matrices'

BASELINE_SEQ  = f'{BASE_DIR}/sequences_baseline.pkl'
ENRICHED_SEQ  = f'{BASE_DIR}/sequences_enriched.pkl'

BASELINE_SAVE = '/content/drive/MyDrive/RAIM_Project_v2/run_05_age_med_years_med/checkpoints_baseline'
ENRICHED_SAVE = '/content/drive/MyDrive/RAIM_Project_v2/run_05_age_med_years_med/checkpoints_enriched'

In [8]:
#Cell 7: Run both and print comparison
results = []

results.append(train_run(
    sequences_path=BASELINE_SEQ,
    input_dim=69,
    save_dir=BASELINE_SAVE,
    label='Baseline (no LLM)'
))

results.append(train_run(
    sequences_path=ENRICHED_SEQ,
    input_dim=340,
    save_dir=ENRICHED_SAVE,
    label='Enriched (LLM ICD)'
))

print("\n" + "="*50)
print(f"{'Model':<25} {'Test RMSE':>12} {'Test MAE':>12}")
print("-"*50)
for r in results:
    print(f"{r['label']:<25} {r['test_rmse']:>12.4f} {r['test_mae']:>12.4f}")
print("="*50)

Split — train: 1935, val: 257, test: 386

Running: Baseline (no LLM)  |  input_dim=69
Epoch 001 | Train RMSE: 10.2574  MAE: 5.8035 | Val RMSE: 5.9394  MAE: 4.4690
  -> Saved best (val RMSE: 5.9394)
Epoch 002 | Train RMSE: 9.9548  MAE: 5.7023 | Val RMSE: 5.9025  MAE: 4.5434
  -> Saved best (val RMSE: 5.9025)
Epoch 003 | Train RMSE: 9.9557  MAE: 5.6461 | Val RMSE: 5.9256  MAE: 4.7457
Epoch 004 | Train RMSE: 9.8735  MAE: 5.8824 | Val RMSE: 5.8955  MAE: 4.6502
  -> Saved best (val RMSE: 5.8955)
Epoch 005 | Train RMSE: 9.9042  MAE: 5.6535 | Val RMSE: 5.9166  MAE: 4.7788
Epoch 006 | Train RMSE: 9.7883  MAE: 5.6360 | Val RMSE: 5.7146  MAE: 4.2818
  -> Saved best (val RMSE: 5.7146)
Epoch 007 | Train RMSE: 9.6234  MAE: 5.4713 | Val RMSE: 5.5920  MAE: 4.2839
  -> Saved best (val RMSE: 5.5920)
Epoch 008 | Train RMSE: 9.5304  MAE: 5.3689 | Val RMSE: 5.5027  MAE: 4.2134
  -> Saved best (val RMSE: 5.5027)
Epoch 009 | Train RMSE: 9.4599  MAE: 5.2674 | Val RMSE: 5.5466  MAE: 4.3385
Epoch 010 | Train R